# Building a Streaming Chatbot with Any-LLM


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mozilla-ai/any-llm/blob/main/docs/cookbooks/any_llm_streaming_chat_interface.ipynb)

In this cookbook, we will create a simple, interactive CLI (Command Line Interface) chatbot that streams responses token-by-token and maintains conversation history.

## What is Streaming?
Streaming is the practice of handling response chunks generated by an LLM instead of waiting for an entire block of text. We'll also reference Context Windows, which is managing a list of messages so the model remembers the conversation.

LLMs generate tokens sequentially, and you typically need to wait for the entire response to complete before seeing anything. This can can feel like staring at a blank screen for a while. With streaming, you can show each token as it's generated by the LLM, making the interaction feel instant and responsive even though the total generation time is identical.

## Installation

In [ ]:
%pip install any-llm-sdk[all] nest-asyncio --quiet

# nest_asyncio allows us to use 'await' directly in Jupyter notebooks
# This is needed because any-llm uses async functions for API calls
import nest_asyncio

nest_asyncio.apply()

## Setting Up API Keys
Let's set up provider keys the right way

In [ ]:
import os
from getpass import getpass


def setup_api_key(key_name: str, provider: str) -> None:
    """Set up API key for the specified provider."""
    if key_name not in os.environ:
        print(f"🔑 {key_name} not found in environment")
        api_key = getpass(f"Enter your {provider} API key (or press Enter to skip): ")
        if api_key:
            os.environ[key_name] = api_key
            print(f"✅ {key_name} set for this session")
        else:
            print(f"⏭️  Skipping {provider}")
    else:
        print(f"✅ {key_name} found in environment")


# Set up keys for different providers
print("Setting up API keys...\n")
setup_api_key("OPENAI_API_KEY", "OpenAI")


## A Single Streaming Request
In the code below, we learn how to handle a single streamed response. Note that the response from `acompletion` is a generator, and not a static string. We then iterate over the chunks sent by the LLM, and then finally extract the content we care about. 

In [ ]:
# Conceptual snippet for the outline
from any_llm import acompletion

messages = [{"role": "user", "content": "Write a haiku about Python."}]
    
# The 'stream=True' is the key addition here
response = await acompletion(
    model="openai:gpt-4o-mini", 
    messages=messages, 
    stream=True
)

print("Assistant: ", end="")

async for chunk in response:
    # any-llm normalizes this access pattern
    content = chunk.choices[0].delta.content
    if content:
        print(content, end="", flush=True)

## Managing Conversation History
Since LLMs are stateless, we may need to manage history ourseleves for certain use cases. We manage conversation history below by simulating a multi-turn conversation. The code below also prints the "Memory State" after each turn so you can see exactly what the model sees.
> Note: For this step, we are using standard (non-streaming) completion to keep the code simple. This allows us to focus entirely on how the messages list is updated. We will bring streaming back in the final step!

In [ ]:
import asyncio
from any_llm import acompletion

# --- The Core Concept: The "Context Window" ---
# LLMs don't remember you. You must send the entire conversation history 
# with every single request. We store this in a list called 'messages'.

async def run_conversation_manager():
    # 1. Initialize Memory
    # Start with a "System" message to define behavior.
    messages = [
        {"role": "system", "content": "You are a helpful AI assistant."}
    ]
    
    print("--- 🧠 Memory Initialized ---")
    print(f"Current Context: {messages}\n")

    # 2. Simulate Turn 1: User asks a question
    user_input_1 = "Hi, my name is Alice."
    print(f"👤 User: {user_input_1}")
    
    # Update Memory
    messages.append({"role": "user", "content": user_input_1})
    
    # Call Model (sending full history)
    response_1 = await acompletion(
        model="openai:gpt-4o-mini", 
        messages=messages
    )
    
    ai_reply_1 = response_1.choices[0].message.content
    print(f"🤖 AI:   {ai_reply_1}")
    
    # CRITICAL STEP: Save AI's reply to memory
    messages.append({"role": "assistant", "content": ai_reply_1})
    
    print(f"\n--- 🧠 Memory State after Turn 1 ---")
    # Pretty print the list to show what the model now "knows"
    for msg in messages:
        print(f"[{msg['role'].upper()}]: {msg['content']}")
    print("-" * 40 + "\n")


    # 3. Simulate Turn 2: User asks a follow-up
    # The model can only answer this because we send the history from Turn 1.
    user_input_2 = "What is my name?" 
    print(f"👤 User: {user_input_2}")
    
    # Update Memory again
    messages.append({"role": "user", "content": user_input_2})
    
    # Call Model (sending updated history)
    response_2 = await acompletion(
        model="openai:gpt-4o-mini", 
        messages=messages
    )
    ai_reply_2 = response_2.choices[0].message.content
    print(f"🤖 AI:   {ai_reply_2}")
    
    # Update Memory again
    messages.append({"role": "assistant", "content": ai_reply_2})
    
    print(f"\n--- 🧠 Final Memory State ---")
    print(f"Total messages stored: {len(messages)}")

# Run the async function
if __name__ == "__main__":
    await run_conversation_manager()

## The Chat Loop
Now, let's put it all together. We will build an infinite loop that keeps the application running until you decide to quit.

This step combines the Streaming mechanics we learned in Step 1 with the Memory Management from Step 2.

Key mechanics to notice in the code:

- **The Accumulator**: As we stream chunks to the screen for you to read instantly, we simultaneously rebuild the complete string in a hidden variable (full_reply_content).

- **State Updates**: Once the stream finishes, we save that accumulated string to our messages list. This ensures the AI remembers what it just said for your next turn.

- **Exit Condition**: The loop checks if you type 'exit' or 'quit' to stop the program safely.

In [ ]:
import asyncio
from any_llm import acompletion

# --- Step 3: The Infinite Chat Loop ---
async def start_chat_session():
    # 1. Initialize Memory
    messages = [
        {"role": "system", "content": "You are a helpful and concise assistant."}
    ]
    
    print("💬 Chat initialized. Type 'exit' or 'quit' to stop.")
    print("-" * 50)

    # 2. Start the Loop
    while True:
        # A. Get User Input
        # We use standard input(). In a real app, this might come from a UI.
        user_input = input("\n👤 You: ")
        
        # Check for exit condition
        if user_input.strip().lower() in ["exit", "quit"]:
            print("\n👋 Ending session. Goodbye!")
            break
            
        # B. Update Memory (User Turn)
        messages.append({"role": "user", "content": user_input})
        
        # C. Call Model with Streaming
        # We use 'stream=True' so we get chunks instead of waiting for the full text.
        try:
            response = await acompletion(
                model="openai:gpt-4o-mini", # Switch to "anthropic:claude-3-haiku" to test provider swapping!
                messages=messages,
                stream=True
            )
            
            # D. Process the Stream
            print("🤖 AI: ", end="")
            
            full_reply_content = ""
            
            async for chunk in response:
                # Extract the tiny piece of text from this chunk
                # any-llm normalizes this structure across all providers
                delta = chunk.choices[0].delta.content
                
                if delta:
                    # Print immediately to screen (Typewriter effect)
                    print(delta, end="", flush=True)
                    
                    # Accumulate for memory storage
                    full_reply_content += delta
            
            # Print a newline to clean up the display
            print() 
            
            # E. Update Memory (AI Turn)
            # Crucial: We must store the *accumulated* text so the model remembers it next time.
            messages.append({"role": "assistant", "content": full_reply_content})
            
        except Exception as e:
            print(f"\n❌ Error: {str(e)}")
            break

# Run the chat loop
if __name__ == "__main__":
    await start_chat_session()

## Next Steps: A User Challenge
Now that we have used `acompletion` and streaming chat, change the model from `openai:gpt-4o-mini` to a model from another cloud or local model. Run the same loop again. 
You will realize that **zero** logic changes were needed to handle the different streaming protocols of these providers—any-llm normalized the chunks for you.